# Summary

Explore the RAG evaluation datasets

In [1]:
import os, sys
import pandas as pd
import json
import time

# AWS Python
import boto3

# Numantic utilities
utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")


## Create a Bedrock Knowlege Base

In [2]:
"""
Complete Guide to Creating an Amazon Bedrock Knowledge Base
============================================================

Prerequisites:
1. S3 bucket with .txt files containing document text
2. AWS credentials configured
3. Python packages: boto3, opensearchpy

This script assumes .txt files are already uploaded to S3
"""


'\nComplete Guide to Creating an Amazon Bedrock Knowledge Base\n============================================================\n\nPrerequisites:\n1. S3 bucket with .txt files containing document text\n2. AWS credentials configured\n3. Python packages: boto3, opensearchpy\n\nThis script assumes .txt files are already uploaded to S3\n'

## Set up configuration and initialize

In [3]:

# ============================================================================
# SECTION 0: Configuration and Initialization
# ============================================================================
# --- Configuration ---
REGION_NAME = 'us-east-2'
BUCKET_NAME = 'ashoka-search-tests'
S3_PREFIX = ''
COLLECTION_NAME = 'ashoka-kb-collection'
KB_NAME = 'ashoka-search-kb'
INDEX_NAME = 'bedrock-knowledge-base-default-index'
ROLE_NAME = 'AmazonBedrockExecutionRoleForKnowledgeBase'
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'

# Use Admin for all Setup tasks to avoid AccessDenied
admin_session = boto3.Session(profile_name='ns-admin')
aoss_client = admin_session.client('opensearchserverless', region_name=REGION_NAME)
iam_client = admin_session.client('iam')
sts_client = admin_session.client('sts')
bedrock_agent = admin_session.client('bedrock-agent', region_name=REGION_NAME)

account_id = sts_client.get_caller_identity()['Account']

## Create Security Policies

In [4]:
# --- Create OpenSearch Serverless Security Policies ---
def create_aoss_policies():
    # Encryption & Network
    aoss_client.create_security_policy(name=f'{COLLECTION_NAME}-enc', type='encryption',
        policy=json.dumps({"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}], "AWSOwnedKey": True}))

    aoss_client.create_security_policy(name=f'{COLLECTION_NAME}-net', type='network',
        policy=json.dumps([{"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}, {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]}], "AllowFromPublic": True}]))

    # Data Access: Grant permissions to the role Bedrock will use
    aoss_client.create_access_policy(name=f'{COLLECTION_NAME}-acc', type='data',
        policy=json.dumps([{
            "Rules": [
                {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"], "Permission": ["aoss:*"]},
                {"ResourceType": "index", "Resource": [f"index/{COLLECTION_NAME}/*"], "Permission": ["aoss:*"]}
            ],
            "Principal": [f"arn:aws:iam::{account_id}:role/{ROLE_NAME}", sts_client.get_caller_identity()['Arn']]
        }]))

# --- Create IAM Role for Bedrock ---
def create_execution_role():
    trust_pol = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
    role_arn = iam_client.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_pol))['Role']['Arn']

    exec_pol = {
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["s3:GetObject", "s3:ListBucket"], "Resource": [f"arn:aws:s3:::{BUCKET_NAME}", f"arn:aws:s3:::{BUCKET_NAME}/*"]},
            {"Effect": "Allow", "Action": ["aoss:APIAccessAll"], "Resource": ["*"]},
            {"Effect": "Allow", "Action": ["bedrock:InvokeModel"], "Resource": [f"arn:aws:bedrock:{REGION_NAME}::foundation-model/{EMBEDDING_MODEL}"]}
        ]
    }
    iam_client.put_role_policy(RoleName=ROLE_NAME, PolicyName='BedrockPolicy', PolicyDocument=json.dumps(exec_pol))
    return role_arn

try:
    create_aoss_policies()
    role_arn = create_execution_role()
    print("✓ Security roles and policies established.")
    time.sleep(10) # Wait for IAM propagation
except Exception as e:
    print(f"Note: Role/Policies might already exist: {e}")
    role_arn = f"arn:aws:iam::{account_id}:role/{ROLE_NAME}"

Note: Role/Policies might already exist: An error occurred (ConflictException) when calling the CreateSecurityPolicy operation: Given encryption policy is conflicting with the existing encryption policies


## Set up Knowledge Base

In [5]:
### Delete knowledge base if it already exists

print(f"Checking for existing Knowledge Base named '{KB_NAME}'...")
paginator = bedrock_agent.get_paginator('list_knowledge_bases')
for page in paginator.paginate():
    for kb in page['knowledgeBaseSummaries']:
        if kb['name'] == KB_NAME:
            print(f"Found existing KB {kb['knowledgeBaseId']}. Deleting...")
            bedrock_agent.delete_knowledge_base(knowledgeBaseId=kb['knowledgeBaseId'])
            # Wait for deletion to propagate
            time.sleep(10)

Checking for existing Knowledge Base named 'ashoka-search-kb'...
Found existing KB UVKTQ7XQAE. Deleting...


In [6]:
# ============================================================================
# SECTION 2: Set up Knowledge Base
# ============================================================================
# --- Create OpenSearch Collection ---
try:
    coll = aoss_client.create_collection(name=COLLECTION_NAME, type='VECTORSEARCH')
    coll_arn = coll['createCollectionDetail']['arn']
    print("Waiting for collection to activate...")
    while aoss_client.batch_get_collection(names=[COLLECTION_NAME])['collectionDetails'][0]['status'] != 'ACTIVE':
        time.sleep(5)
except Exception:
    coll_arn = aoss_client.batch_get_collection(names=[COLLECTION_NAME])['collectionDetails'][0]['arn']

# --- Create Knowledge Base ---
kb_response = bedrock_agent.create_knowledge_base(
    name=KB_NAME,
    roleArn=role_arn,
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration': {
            'embeddingModelArn': f'arn:aws:bedrock:{REGION_NAME}::foundation-model/{EMBEDDING_MODEL}'
        }
    },
    storageConfiguration={
        'type': 'OPENSEARCH_SERVERLESS',
        'opensearchServerlessConfiguration': {
            'collectionArn': coll_arn,
            'vectorIndexName': INDEX_NAME,
            'fieldMapping': {
                'vectorField': 'bedrock-knowledge-base-default-vector',
                'textField': 'AMAZON_BEDROCK_TEXT_CHUNK',
                'metadataField': 'AMAZON_BEDROCK_METADATA' # This enables filtering!
            }
        }
    }
)
kb_id = kb_response['knowledgeBase']['knowledgeBaseId']
print(f"✓ Knowledge Base Created: {kb_id}")

✓ Knowledge Base Created: DBSECKYXJV


## Ingest documents to the Knowledge Base

In [8]:
# ============================================================================
# SECTION 3: Ingest Source Data
# ============================================================================
# --- Add S3 Data Source ---
ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name='s3-docs-with-metadata',
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': f'arn:aws:s3:::{BUCKET_NAME}',
            # 'inclusionPrefixes': [S3_PREFIX]
        }
    }
)
ds_id = ds_response['dataSource']['dataSourceId']

# --- Start Ingestion ---
ingestion_response = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
print(f"✓ Ingestion started for {BUCKET_NAME}/{S3_PREFIX}")
print("You can now filter by 'source_type' or 'doc_index' in your queries!")

✓ Ingestion started for ashoka-search-tests/
You can now filter by 'source_type' or 'doc_index' in your queries!


In [9]:
dir(bedrock_agent)

['_PY_TO_OP_NAME',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_cache',
 '_client_config',
 '_convert_to_request_dict',
 '_emit_api_params',
 '_endpoint',
 '_exceptions',
 '_exceptions_factory',
 '_get_credentials',
 '_get_waiter_config',
 '_load_exceptions',
 '_loader',
 '_make_api_call',
 '_make_request',
 '_register_handlers',
 '_request_signer',
 '_resolve_endpoint_ruleset',
 '_response_parser',
 '_ruleset_resolver',
 '_serializer',
 '_service_model',
 '_user_agent_creator',
 'associate_agent_collaborator',
 'associate_agent_knowledge_base',
 'can_paginate',
 'close',
 'create_agent

## Check status of ingestion job

In [10]:
ingestion_job_id = ingestion_response['ingestionJob']['ingestionJobId']
print(f"Monitoring Ingestion Job: {ingestion_job_id}...")

while True:
    job_response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=ingestion_job_id
    )

    status = job_response['ingestionJob']['status']
    print(f"Current Status: {status}")

    if status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break

    time.sleep(10)

# --- Detailed Reporting ---
job_data = job_response['ingestionJob']

print("\n" + "="*30)
print("INGESTION JOB SUMMARY")
print("="*30)
print(f"Status: {job_data['status']}")

# Print Statistics if available
if 'statistics' in job_data:
    stats = job_data['statistics']
    print(f"Documents Scanned: {stats.get('numberOfDocumentsScanned', 0)}")
    print(f"Documents Indexed: {stats.get('numberOfNewDocumentsIndexed', 0)}")
    print(f"Documents Failed:  {stats.get('numberOfDocumentsFailed', 0)}")

# Print Failure Reasons
if 'failureReasons' in job_data:
    print("\nFailure Reasons:")
    for reason in job_data['failureReasons']:
        print(f"  - {reason}")

# Full Debugging Dump
print("\nFull JSON Job Details (for debugging):")
print(json.dumps(job_data, indent=2, default=str))


Monitoring Ingestion Job: ZPYUPWKGMA...
Current Status: IN_PROGRESS
Current Status: IN_PROGRESS
Current Status: COMPLETE

INGESTION JOB SUMMARY
Status: COMPLETE
Documents Scanned: 20
Documents Indexed: 20
Documents Failed:  0

Full JSON Job Details (for debugging):
{
  "knowledgeBaseId": "DBSECKYXJV",
  "dataSourceId": "2GCPNEX1JG",
  "ingestionJobId": "ZPYUPWKGMA",
  "status": "COMPLETE",
  "statistics": {
    "numberOfDocumentsScanned": 20,
    "numberOfMetadataDocumentsScanned": 20,
    "numberOfNewDocumentsIndexed": 20,
    "numberOfModifiedDocumentsIndexed": 0,
    "numberOfMetadataDocumentsModified": 0,
    "numberOfDocumentsDeleted": 0,
    "numberOfDocumentsFailed": 0
  },
  "startedAt": "2026-04-02 21:54:07.784319+00:00",
  "updatedAt": "2026-04-02 21:54:32.524462+00:00"
}
